# Scraping Ulasan Steam: *Lies of P*

**Proyek Analisis Sentimen — Dicoding**

Notebook ini mengumpulkan ulasan pemain game *Lies of P* (Steam App ID `1627720`) secara mandiri untuk analisis sentimen. Data hasil scraping diolah lebih lanjut pada notebook `pelatihan_model_sentimen.ipynb`.

**Sumber data dan etika pengambilan data:**
- Data diambil dari endpoint publik resmi Steam `store.steampowered.com/appreviews/<appid>?json=1`, yang memang disediakan Steam untuk membaca ulasan toko. Tidak ada login maupun upaya melewati proteksi.
- Setiap permintaan diberi jeda 2 detik agar tidak membebani server.
- **Privasi:** identitas penulis ulasan (SteamID, nama profil) tidak diambil maupun disimpan.
- Hanya ulasan berbahasa Inggris yang diambil, karena jumlahnya paling banyak dan pelabelan dengan lexicon VADER dirancang untuk bahasa Inggris.

## 1. Import library
- `os`: memeriksa keberadaan file checkpoint.
- `time`: memberi jeda antar permintaan.
- `requests`: mengirim permintaan ke server Steam.
- `pandas`: mengolah data menjadi tabel dan menyimpannya ke CSV.

In [ ]:
import os
import time
import requests
import pandas as pd

requests : 2.32.4
pandas   : 2.2.3


## 2. Konfigurasi
Seluruh pengaturan dikumpulkan di satu sel. Target 20.000 ulasan mentah dipilih agar setelah pembersihan (membuang ulasan kosong, sangat pendek, duplikat, dan bukan bahasa Inggris) data tetap di atas 10.000 sampel.

In [ ]:
APP_ID = 1627720            # ID game Lies of P di Steam
TARGET_JUMLAH = 20000       # target ulasan mentah
JEDA_DETIK = 2.0            # jeda antar request (detik)

OUTPUT_CSV = 'ulasan_lies_of_p.csv'          # file hasil akhir
CHECKPOINT_CSV = 'checkpoint_ulasan.csv'     # simpanan progres sementara
CHECKPOINT_CURSOR = 'checkpoint_cursor.txt'  # posisi halaman terakhir

URL = f'https://store.steampowered.com/appreviews/{APP_ID}'
HEADERS = {'User-Agent': 'Mozilla/5.0 (proyek-analisis-sentimen-dicoding)'}

print('URL endpoint:', URL)

URL endpoint: https://store.steampowered.com/appreviews/1627720


## 3. Fungsi pengambilan satu halaman
Setiap permintaan menghasilkan maksimal 100 ulasan beserta `cursor` untuk halaman berikutnya. Parameter yang digunakan:

| Parameter | Nilai | Keterangan |
|---|---|---|
| `filter` | `recent` | urut dari yang terbaru, dapat ditelusuri dengan cursor hingga ulasan habis |
| `language` | `english` | hanya ulasan berbahasa Inggris |
| `review_type` | `all` | ulasan positif dan negatif |
| `purchase_type` | `all` | semua pembeli |
| `num_per_page` | 100 | jumlah maksimum per halaman |

Steam membatasi jumlah permintaan dalam rentang waktu tertentu dan membalas dengan **HTTP 429 (Too Many Requests)** bila batas terlampaui. Fungsi ini menunggu minimal 60 detik (lalu 120, 180, dan seterusnya) saat menerima 429 dan mencoba ulang hingga 6 kali. Jika tetap gagal, fungsi mengembalikan `None` alih-alih menghentikan program, sehingga data yang sudah terkumpul tidak hilang.

In [ ]:
def ambil_halaman(cursor, percobaan=6):
    params = {
        'json': 1,
        'filter': 'recent',
        'language': 'english',
        'review_type': 'all',
        'purchase_type': 'all',
        'num_per_page': 100,
        'cursor': cursor,
    }
    for i in range(percobaan):
        try:
            r = requests.get(URL, params=params, headers=HEADERS, timeout=30)

            if r.status_code == 429:
                retry_after = r.headers.get('Retry-After', '')
                tunggu = int(retry_after) if retry_after.isdigit() else 0
                tunggu = max(tunggu, 60 * (i + 1))
                print(f'  429 Too Many Requests, menunggu {tunggu} detik (percobaan {i + 1}/{percobaan})')
                time.sleep(tunggu)
                continue

            r.raise_for_status()
            data = r.json()
            if data.get('success') == 1:
                return data
        except (requests.RequestException, ValueError) as e:
            print(f'  percobaan {i + 1} gagal: {e}')
            time.sleep(5 * 2 ** i)
    return None

### 3.1 Uji coba satu halaman
Sebelum mengambil seluruh data, fungsi diuji dengan satu permintaan.

In [ ]:
uji = ambil_halaman('*')

if uji is None:
    print('Gagal, kemungkinan masih dibatasi Steam.')
else:
    print('Ringkasan:', uji['query_summary'])
    print('Jumlah ulasan di halaman ini:', len(uji['reviews']))
    print('Cursor berikutnya:', uji['cursor'])
    print('\nContoh ulasan pertama:')
    print(uji['reviews'][0]['review'][:300])

Ringkasan: {'num_reviews': 100, 'review_score': 8, 'review_score_desc': 'Very Positive', 'total_positive': 35250, 'total_negative': 2576, 'total_reviews': 37826}
Jumlah ulasan di halaman ini: 100
Cursor berikutnya: AoJ4yqmml6EDeu6Kigc=

Contoh ulasan pertama:
Meeting boss means:
You start attacking -> 1 attack and you need to parry, boss starts charging as soon as he sees you pressing left button
You don't attack? -> Boss won't be attacking either! Just practice in ballet with you, how nice of developers!
Parry slop
Funny to see people sometimes praise d


**Hasil:** permintaan berhasil dan mengembalikan 100 ulasan beserta cursor halaman berikutnya. Ringkasan dari Steam menunjukkan total 37.826 ulasan berbahasa Inggris (35.250 positif dan 2.576 negatif), sehingga target 20.000 ulasan dapat terpenuhi. Data sangat timpang ke arah positif (93,2%).

Contoh ulasan pertama bernada sarkastis ("how nice of developers!"), gambaran awal tantangan pelabelan otomatis berbasis kamus kata.

## 4. Scraping dengan checkpoint
Fungsi `scrape_ulasan` mengulang pengambilan halaman hingga target tercapai. Progres disimpan ke file checkpoint setiap 10 halaman (1.000 ulasan), sehingga jika proses terhenti, misalnya karena pembatasan Steam, scraping dapat dilanjutkan dari posisi terakhir tanpa mengulang dari awal. Sebagai pengaman, loop juga berhenti jika Steam mengembalikan cursor yang sama (ulasan sudah habis).

Kolom yang disimpan: ID ulasan, teks ulasan, status rekomendasi (`voted_up`), jumlah vote membantu, jam bermain saat menulis ulasan, waktu pembuatan, dan bahasa.

In [ ]:
def simpan_checkpoint(hasil, cursor):
    pd.DataFrame(hasil).to_csv(CHECKPOINT_CSV, index=False)
    with open(CHECKPOINT_CURSOR, 'w') as f:
        f.write(cursor)


def muat_checkpoint():
    if os.path.exists(CHECKPOINT_CSV) and os.path.exists(CHECKPOINT_CURSOR):
        hasil = pd.read_csv(CHECKPOINT_CSV).to_dict('records')
        with open(CHECKPOINT_CURSOR) as f:
            cursor = f.read().strip()
        print(f'Melanjutkan dari checkpoint: {len(hasil)} ulasan sudah terkumpul.')
        return hasil, cursor
    return [], '*'


def scrape_ulasan(target):
    hasil, cursor = muat_checkpoint()
    cursor_terpakai, halaman = set(), 0

    while len(hasil) < target:
        data = ambil_halaman(cursor)
        if data is None:
            simpan_checkpoint(hasil, cursor)
            print(f'Berhenti sementara. {len(hasil)} ulasan aman di checkpoint. '
                  'Tunggu 10-15 menit lalu jalankan ulang sel ini.')
            break

        ulasan = data.get('reviews', [])
        if not ulasan:
            print('Tidak ada ulasan lagi.')
            break

        for u in ulasan:
            hasil.append({
                'review_id': u.get('recommendationid'),
                'review': u.get('review'),
                'voted_up': u.get('voted_up'),
                'votes_up': u.get('votes_up'),
                'playtime_at_review_min': u.get('author', {}).get('playtime_at_review'),
                'timestamp_created': u.get('timestamp_created'),
                'language': u.get('language'),
            })

        halaman += 1
        cursor_terpakai.add(cursor)
        cursor = data.get('cursor')
        if not cursor or cursor in cursor_terpakai:
            print('Cursor habis, semua ulasan sudah terambil.')
            break

        if halaman % 10 == 0:
            simpan_checkpoint(hasil, cursor)
            print(f'Terkumpul: {len(hasil)} ulasan (checkpoint disimpan)')

        time.sleep(JEDA_DETIK)

    simpan_checkpoint(hasil, cursor or '*')
    return hasil

### 4.1 Menjalankan scraping

In [ ]:
data_mentah = scrape_ulasan(TARGET_JUMLAH)
df = pd.DataFrame(data_mentah)
print('Total ulasan mentah:', len(df))
df.head()

Terkumpul: 1000 ulasan (checkpoint disimpan)
Terkumpul: 2000 ulasan (checkpoint disimpan)
Terkumpul: 3000 ulasan (checkpoint disimpan)
Terkumpul: 4000 ulasan (checkpoint disimpan)
Terkumpul: 5000 ulasan (checkpoint disimpan)
Terkumpul: 6000 ulasan (checkpoint disimpan)
Terkumpul: 7000 ulasan (checkpoint disimpan)
Terkumpul: 8000 ulasan (checkpoint disimpan)
Terkumpul: 9000 ulasan (checkpoint disimpan)
Terkumpul: 10000 ulasan (checkpoint disimpan)
Terkumpul: 10999 ulasan (checkpoint disimpan)
Terkumpul: 11999 ulasan (checkpoint disimpan)
Terkumpul: 12999 ulasan (checkpoint disimpan)
Terkumpul: 13999 ulasan (checkpoint disimpan)
Terkumpul: 14999 ulasan (checkpoint disimpan)
Terkumpul: 15999 ulasan (checkpoint disimpan)
Terkumpul: 16999 ulasan (checkpoint disimpan)
Terkumpul: 17999 ulasan (checkpoint disimpan)
Terkumpul: 18999 ulasan (checkpoint disimpan)
Terkumpul: 19998 ulasan (checkpoint disimpan)
Total ulasan mentah: 20098


,review_id,review,voted_up,votes_up,playtime_at_review_min,timestamp_created,language
0,238073815,Meeting boss means:\nYou start attacking -> 1 ...,False,0,461,1792386875,english
1,238072758,P lied to me so far,True,0,1568,1792384936,english
2,238072602,"I absolutely recommend this game, with a cavea...",True,0,5124,1792384684,english
3,238072322,Peak,True,0,1721,1792384213,english
4,238066578,great game -- really enjoying it so far.,True,0,1412,1792375384,english


**Hasil:** terkumpul 20.098 ulasan mentah tanpa kendala pembatasan dari Steam. Jumlahnya sedikit melebihi target karena target diperiksa setelah satu halaman penuh (100 ulasan) masuk.

## 5. Pengecekan kualitas data dan penyimpanan
Duplikat berdasarkan ID ulasan dihapus (antisipasi halaman yang terambil dua kali saat melanjutkan dari checkpoint), lalu waktu ulasan diubah dari Unix timestamp ke format tanggal.

In [ ]:
df = df.drop_duplicates(subset='review_id')
df['tanggal'] = pd.to_datetime(df['timestamp_created'], unit='s')

print('Jumlah setelah hapus duplikat ID :', len(df))
print('Rentang tanggal ulasan           :', df['tanggal'].min().date(), 's.d.', df['tanggal'].max().date())
print('Proporsi direkomendasikan        :', round(df['voted_up'].mean(), 3))
print('Ulasan kosong                    :', df['review'].isna().sum() + (df['review'].fillna('').str.strip() == '').sum())

df.to_csv(OUTPUT_CSV, index=False)
print(f'\nTersimpan ke {OUTPUT_CSV}')

Jumlah setelah hapus duplikat ID : 20098
Rentang tanggal ulasan           : 2024-07-16 s.d. 2026-10-19
Proporsi direkomendasikan        : 0.922
Ulasan kosong                    : 76

Tersimpan ke ulasan_lies_of_p.csv


**Hasil:** tidak ada ID ganda (20.098 ulasan). Ulasan berasal dari 16 Juli 2024 hingga 19 Oktober 2026, 92,2% merekomendasikan game, dan 76 ulasan tidak berisi teks. Pembersihan lebih lanjut dilakukan pada notebook pelatihan.

### 5.1 Merapikan format CSV
Baris baru (Enter) di dalam teks ulasan diganti spasi dan file disimpan ulang dengan encoding `utf-8-sig`, agar setiap ulasan menempati satu baris ketika dibuka di aplikasi spreadsheet (Excel/WPS). Perubahan ini tidak mengubah isi ulasan. Jumlah baris dalam file diverifikasi sama dengan data di memori.

In [ ]:
df['review'] = (df['review']
                .fillna('')
                .str.replace(r'[\r\n\t]+', ' ', regex=True)
                .str.strip())

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

cek = pd.read_csv(OUTPUT_CSV)
print('Baris di memori :', len(df))
print('Baris di file   :', len(cek))
print('Ulasan masih mengandung Enter:', cek['review'].fillna('').str.contains('\n').sum())

Baris di memori : 20098
Baris di file   : 20098
Ulasan masih mengandung Enter: 0


### 5.2 Mengunduh dataset
File `ulasan_lies_of_p.csv` diunduh untuk disertakan dalam berkas submission.

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>